# [OUTDATED] Silver Layer - pump.fun Trades and Tokens

> **Superseded by the three Silver transformations in `pumpapi-lakehouse/transformations/`** (`silver_pump_events.py`, `silver_pump_tokens.py`, `silver_pump_transfers.py`), part of the same Lakeflow Declarative Pipeline as Bronze. Kept for reference only -- do not deploy.
>
> This notebook's `silver_pumpfun_trades` / `silver_pumpfun_tokens` design was replaced by a different table split (all events / token creations / transfers) chosen when the pipeline was rebuilt as a single Lakeflow pipeline.

---

# Silver Layer - pump.fun Trades and Tokens

Reads `bronze.pumpfun_events` incrementally and parses two typed Silver
tables out of the raw per-action JSON:

- **`silver_pumpfun_trades`** — append-only fact table of `buy`/`sell`
  events (one row per trade side per pool).
- **`silver_pumpfun_tokens`** — current-state table keyed by `mint`,
  upserted from `create`/`migrate` lifecycle events (tracks mint/freeze
  authority, which pool type it's trading on, and migration status —
  useful rug-pull risk signals per the PumpAPI glossary).

Other action types (`createPool`, `add`, `remove`, `transfer`,
`claimCashback`, `claimCreatorFees`) are left in Bronze only for now; see
`ROADMAP.md` for follow-up scope.

In [ ]:
from pyspark.sql.functions import col, get_json_object, current_timestamp, to_timestamp
from delta.tables import DeltaTable

dbutils.widgets.text("CATALOG",         "workspace")
dbutils.widgets.text("BRONZE_SCHEMA",   "bronze")
dbutils.widgets.text("SILVER_SCHEMA",   "silver")
dbutils.widgets.text("CHECKPOINT_PATH", "/Volumes/workspace/default/mnt/checkpoints/silver_pumpfun")

CONFIG = {
    "catalog":         dbutils.widgets.get("CATALOG"),
    "bronze_schema":   dbutils.widgets.get("BRONZE_SCHEMA"),
    "silver_schema":   dbutils.widgets.get("SILVER_SCHEMA"),
    "checkpoint_path": dbutils.widgets.get("CHECKPOINT_PATH"),
}

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.pumpfun_events"
trades_table_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}.silver_pumpfun_trades"
tokens_table_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}.silver_pumpfun_tokens"

TRADE_ACTIONS = ("buy", "sell")
TOKEN_LIFECYCLE_ACTIONS = ("create", "migrate")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['catalog']}.{CONFIG['silver_schema']}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {trades_table_fqn} (
        signature           STRING    NOT NULL,
        action               STRING    NOT NULL,
        mint                 STRING,
        pool_id              STRING,
        quote_mint           STRING,
        pool                 STRING,
        price                DOUBLE,
        token_amount         DOUBLE,
        quote_amount         DOUBLE,
        market_cap_quote     DOUBLE,
        tx_signer            STRING,
        traders_involved     STRING,
        block                LONG,
        event_timestamp      TIMESTAMP,
        bronze_ingested_at   TIMESTAMP,
        silver_processed_at  TIMESTAMP NOT NULL
    ) USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tokens_table_fqn} (
        mint                 STRING    NOT NULL,
        pool_type            STRING,
        pool_created_by      STRING,
        mint_authority       STRING,
        freeze_authority     STRING,
        token_program        STRING,
        created_at           TIMESTAMP,
        migrated_at          TIMESTAMP,
        last_seen_action     STRING,
        last_seen_block      LONG,
        last_updated_dt      TIMESTAMP NOT NULL
    ) USING DELTA
""")

print(f"{trades_table_fqn} is ready")
print(f"{tokens_table_fqn} is ready")

In [ ]:
def _parse_trades(batch_df):
    """buy/sell events -> one row per (signature, action, mint, pool_id)."""
    trades = (
        batch_df
        .filter(col("action").isin(*TRADE_ACTIONS))
        .withColumn("quote_mint",       get_json_object(col("event_json"), "$.quoteMint"))
        .withColumn("pool",             get_json_object(col("event_json"), "$.pool"))
        .withColumn("price",            get_json_object(col("event_json"), "$.price").cast("double"))
        .withColumn("token_amount",     get_json_object(col("event_json"), "$.tokenAmount").cast("double"))
        .withColumn("quote_amount",     get_json_object(col("event_json"), "$.quoteAmount").cast("double"))
        .withColumn("market_cap_quote", get_json_object(col("event_json"), "$.marketCapQuote").cast("double"))
        .withColumn("tx_signer",        get_json_object(col("event_json"), "$.txSigner"))
        .withColumn("traders_involved", get_json_object(col("event_json"), "$.tradersInvolved"))
        .withColumn("block",            get_json_object(col("event_json"), "$.block").cast("long"))
        # PumpAPI 'timestamp' is a blockchain tx timestamp; treat as unix seconds.
        .withColumn("event_timestamp",  to_timestamp(get_json_object(col("event_json"), "$.timestamp").cast("double")))
        .withColumn("silver_processed_at", current_timestamp())
        .select(
            "signature", "action", "mint", "pool_id", "quote_mint", "pool",
            "price", "token_amount", "quote_amount", "market_cap_quote",
            "tx_signer", "traders_involved", "block", "event_timestamp",
            "bronze_ingested_at", "silver_processed_at",
        )
        .dropDuplicates(["signature", "action", "mint", "pool_id"])
    )
    return trades


def _parse_tokens(batch_df):
    """create/migrate events -> latest known state per mint."""
    tokens = (
        batch_df
        .filter(col("action").isin(*TOKEN_LIFECYCLE_ACTIONS))
        .withColumn("pool_type",       get_json_object(col("event_json"), "$.pool"))
        .withColumn("pool_created_by", get_json_object(col("event_json"), "$.poolCreatedBy"))
        .withColumn("mint_authority",  get_json_object(col("event_json"), "$.mintAuthority"))
        .withColumn("freeze_authority", get_json_object(col("event_json"), "$.freezeAuthority"))
        .withColumn("token_program",   get_json_object(col("event_json"), "$.tokenProgram"))
        .withColumn("block",           get_json_object(col("event_json"), "$.block").cast("long"))
        .withColumn("event_timestamp", to_timestamp(get_json_object(col("event_json"), "$.timestamp").cast("double")))
        .select(
            "mint", "action", "pool_type", "pool_created_by", "mint_authority",
            "freeze_authority", "token_program", "block", "event_timestamp",
        )
    )
    return tokens

In [ ]:
def write_silver(batch_df, batch_id):
    if not batch_df.take(1):
        return

    trades = _parse_trades(batch_df)
    if trades.take(1):
        trades_delta = DeltaTable.forName(spark, trades_table_fqn)
        (
            trades_delta.alias("t")
            .merge(
                trades.alias("s"),
                "t.signature = s.signature AND t.action = s.action "
                "AND t.mint = s.mint AND t.pool_id = s.pool_id",
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"  batch {batch_id}: merged {trades.count()} trade rows")

    tokens = _parse_tokens(batch_df)
    if tokens.take(1):
        tokens_delta = DeltaTable.forName(spark, tokens_table_fqn)
        (
            tokens_delta.alias("t")
            .merge(tokens.alias("s"), "t.mint = s.mint")
            .whenMatchedUpdate(set={
                "pool_type":        "coalesce(s.pool_type, t.pool_type)",
                "pool_created_by":  "coalesce(s.pool_created_by, t.pool_created_by)",
                "mint_authority":   "coalesce(s.mint_authority, t.mint_authority)",
                "freeze_authority": "coalesce(s.freeze_authority, t.freeze_authority)",
                "token_program":    "coalesce(s.token_program, t.token_program)",
                "migrated_at":      "CASE WHEN s.action = 'migrate' THEN s.event_timestamp ELSE t.migrated_at END",
                "last_seen_action": "s.action",
                "last_seen_block":  "s.block",
                "last_updated_dt":  "current_timestamp()",
            })
            .whenNotMatchedInsert(values={
                "mint":             "s.mint",
                "pool_type":        "s.pool_type",
                "pool_created_by":  "s.pool_created_by",
                "mint_authority":   "s.mint_authority",
                "freeze_authority": "s.freeze_authority",
                "token_program":    "s.token_program",
                "created_at":       "CASE WHEN s.action = 'create' THEN s.event_timestamp ELSE NULL END",
                "migrated_at":      "CASE WHEN s.action = 'migrate' THEN s.event_timestamp ELSE NULL END",
                "last_seen_action": "s.action",
                "last_seen_block":  "s.block",
                "last_updated_dt":  "current_timestamp()",
            })
            .execute()
        )
        print(f"  batch {batch_id}: merged {tokens.count()} token lifecycle rows")

In [ ]:
query = (
    spark.readStream.table(bronze_table_fqn)
    .writeStream
    .foreachBatch(write_silver)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("✅ pump.fun Silver processing completed")